# Four-channel EMG classification prototype

This notebook is the software proof-of-concept for our capstone project. It takes a recorded four-channel EMG signal and follows the main steps from the reference paper: filtering, windowing, feature extraction, LDA, and Gaussian Naive Bayes classification.

> This is an offline engineering prototype. It is not a medical device, and it does not connect to our hardware or control a game yet.

## 1. Load one participant

Download one raw participant MAT file as described in `README.md`, rename it `participant.mat`, and place it in `data/raw/`. Run this notebook from the repository root.

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from emg_pipeline import (
    CLASS_LABELS,
    build_classifier,
    evaluate_classifier,
    extract_features,
    filter_recording,
    load_recording,
    segment_trials,
)

DATA_PATH = Path("data/raw/participant.mat")
recording = load_recording(DATA_PATH)
print(f"Participant: {recording.participant_id}")
print(f"Signal shape: {recording.signal.shape}")
print(f"Sampling rate: {recording.fs} Hz")

## 2. Filter, segment, and view the calibration signals

Following the paper, the full recording is filtered from 20 to 400 Hz before it is split into gesture trials. The plot uses the same general layout as Figure 4(a): movements are shown in columns and EMG channels are shown in rows. The public dataset does not give the channels the same muscle names used in the paper, so we simply call them Channel 1-4.

In [ ]:
filtered_signal = filter_recording(recording.signal, recording.fs)
filtered_recording = replace(recording, signal=filtered_signal)
trials = segment_trials(filtered_recording)
print(f"Extracted trials: {len(trials)} (expected: 30)")

movement_order = [
    "rest",
    "wrist_flexion",
    "wrist_extension",
    "supination",
    "pronation",
    "finger_abduction",
]
display_names = {
    "rest": "Relax / passive pose\nNo input",
    "wrist_flexion": "Wrist flexion\nLeft",
    "wrist_extension": "Wrist extension\nRight",
    "supination": "Supination\nRotate right",
    "pronation": "Pronation\nRotate left",
    "finger_abduction": "Finger abduction\nDrop",
}
movement_colors = {
    "rest": "#D8A100",
    "wrist_flexion": "#F05A28",
    "wrist_extension": "#2E9D47",
    "supination": "#168AAD",
    "pronation": "#7B2CBF",
    "finger_abduction": "#D81B60",
}

fig, axes = plt.subplots(4, 6, figsize=(16, 7), sharex=True)
for column, label in enumerate(movement_order):
    trial = next(t for t in trials if t.repetition == 1 and t.label == label)
    time = np.arange(trial.signal.shape[0]) / recording.fs
    for channel in range(4):
        axis = axes[channel, column]
        axis.plot(
            time[::4],
            trial.signal[::4, channel],
            color=movement_colors[label],
            linewidth=0.45,
        )
        axis.axhline(0, color="0.8", linewidth=0.5)
        axis.spines[["top", "right"]].set_visible(False)
        axis.tick_params(axis="both", labelsize=7)
        if column == 0:
            axis.set_ylabel(f"Channel {channel + 1}\nAmplitude", fontsize=8)
        if channel == 0:
            axis.set_title(display_names[label], color=movement_colors[label], fontsize=10)
        if channel == 3:
            axis.set_xlabel("Time (s)", fontsize=8)

fig.suptitle("Four-channel EMG calibration data - repetition 1", fontsize=14, y=1.02)
fig.tight_layout()

## 3. Extract 16 time-domain features

Each 64 ms window overlaps the previous window by 32 ms. For each of four channels, the code calculates RMS, waveform length, zero crossings, and slope-sign changes.

In [ ]:
feature_set = extract_features(trials, recording.fs)
labels, counts = np.unique(feature_set.y, return_counts=True)

print(f"Feature matrix: {feature_set.X.shape} (expected: (5580, 16))")
print("Windows per class:")
for label, count in zip(labels, counts):
    print(f"  {label:20s} {count}")

## 4. Five-fold LDA + GaussianNB evaluation

The paper uses random five-fold cross-validation, so we use the same approach. In each fold, LDA and Gaussian Naive Bayes are fitted using only the training windows. The table reports the mean and standard deviation across the five folds, as in the paper. One limitation is that adjacent windows overlap, which can make the result look better than it would on a completely new recording.

In [ ]:
evaluation = evaluate_classifier(feature_set.X, feature_set.y)

print("Fold accuracies:", np.round(evaluation.fold_accuracies, 4))
print(f"Mean accuracy: {evaluation.mean_accuracy:.3%} +/- {evaluation.std_accuracy:.3%}")
print("Per-class precision (mean +/- standard deviation):")
for label in evaluation.class_labels:
    mean = evaluation.per_class_precision[label]
    std = evaluation.per_class_precision_std[label]
    print(f"  {label:20s} {mean:.3%} +/- {std:.3%}")

control_name = {
    "rest": "No input",
    "wrist_flexion": "Left",
    "wrist_extension": "Right",
    "supination": "Rotate right",
    "pronation": "Rotate left",
    "finger_abduction": "Drop",
}
metric_headers = ["Accuracy"] + [control_name[label] for label in evaluation.class_labels]
metric_values = [
    f"{100 * evaluation.mean_accuracy:.2f} ({100 * evaluation.std_accuracy:.2f})"
] + [
    (
        f"{100 * evaluation.per_class_precision[label]:.2f} "
        f"({100 * evaluation.per_class_precision_std[label]:.2f})"
    )
    for label in evaluation.class_labels
]

fig, ax = plt.subplots(figsize=(14, 2.2))
ax.axis("off")
table = ax.table(
    cellText=[metric_values],
    colLabels=metric_headers,
    rowLabels=[f"Participant {recording.participant_id}"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)
for (row, column), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#E8F1F5")
ax.set_title("Accuracy and precision metrics: mean (standard deviation), %", pad=16)
fig.tight_layout()

## 5. Visualize the LDA calibration space

Figure 4(c) in the paper shows the first two LDA dimensions. We make the same plot to see which movements form separate clusters. This model is fitted to all windows only for the plot; the accuracy values above still come from cross-validation.

In [ ]:
visualization_model = build_classifier()
visualization_model.fit(feature_set.X, feature_set.y)
lda_projection = visualization_model.named_steps["lda"].transform(feature_set.X)

fig, ax = plt.subplots(figsize=(9, 7))
for label in movement_order:
    mask = feature_set.y == label
    ax.scatter(
        lda_projection[mask, 1],
        lda_projection[mask, 0],
        s=9,
        alpha=0.45,
        color=movement_colors[label],
        label=control_name[label],
        edgecolors="none",
    )

ax.set_xlabel("LDA dimension 2 (a.u.)")
ax.set_ylabel("LDA dimension 1 (a.u.)")
ax.set_title("Calibration data in the two most prominent LDA dimensions")
ax.grid(alpha=0.2)
ax.legend(title="Game control", ncols=2, frameon=False)
fig.tight_layout()

## Interpretation

Our goal at this stage is to show that the complete signal-processing and classification chain works on real four-channel EMG data. The accuracy is based on one healthy participant and overlapping windows, so it should only be treated as a prototype result. It does not tell us how well the system will work for a person with MS, a different sensor placement, or our final hardware.

The paper uses five-window majority voting during live gameplay. This dataset does not include a separate gameplay recording, so we leave that step for the later hardware and game-integration stage instead of creating an artificial live test. The `predict_with_voting` function is included for that stage.